In [18]:
import pandas as pd
import yaml
import numpy as np
from numpy.random import seed
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GridSearchCV

import random

seed(1234)

In [19]:
from doubleml.datasets import fetch_401K

data = fetch_401K(return_type='DataFrame')

print(data.shape)
data

(9915, 14)


,nifa,net_tfa,tw,age,inc,fsize,educ,db,marr,twoearn,e401,p401,pira,hown
0,0.0,0.0,4500.0,47,6765.0,2,8,0,0,0,0,0,0,1
1,6215.0,1015.0,22390.0,36,28452.0,1,16,0,0,0,0,0,0,1
2,0.0,-2000.0,-2000.0,37,3300.0,6,12,1,0,0,0,0,0,0
3,15000.0,15000.0,155000.0,58,52590.0,2,16,0,1,1,0,0,0,1
4,0.0,0.0,58000.0,32,21804.0,1,11,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9910,98498.0,98858.0,157858.0,52,73920.0,1,16,1,0,0,1,1,0,1
9911,287.0,6230.0,15730.0,41,42927.0,4,14,0,1,1,1,1,1,1
9912,99.0,6099.0,7406.0,40,23619.0,2,16,1,0,0,1,0,1,0
9913,0.0,-32.0,2468.0,47,14280.0,4,6,1,0,0,1,1,0,0


In [20]:
outcome = 'net_tfa'
treatment = 'e401' # not 'p401'
confounders=['age', 'inc', 'educ', 'fsize', 'marr', 'twoearn', 'db', 'pira', 'hown']
print(len(confounders))

9


In [21]:
data_bootstrap = data.sample(frac=3, replace=True) # when just using the original data size, a split could have just e410 == 0 or 1
print(data_bootstrap.shape)

(29745, 14)


In [22]:
def train_model(X, y):
    """
    Trains a Random Forest model (classifier or regressor) using GridSearchCV.
    
    Parameters:
        X (pd.DataFrame or np.array): Feature matrix.
        y (pd.Series or np.array): Target variable.
        
    Returns:
        model (RandomForestClassifier or RandomForestRegressor): Trained model.
        params (dict): Best hyperparameters found.
        score (float): Best score from GridSearchCV.
    """
    random_grid = {
        'n_estimators': [50, 100, 150, 200, 250, 300],
        'max_depth': [1, 2, 3, 4, 5, 6]
    }

    # Determine if classification or regression
    print("Unique values in y:", y.unique())
    is_classification = np.issubdtype(y.dtype, np.number) and y.nunique() == 2

    if is_classification:
        rf = RandomForestClassifier()
        scoring_metric = 'accuracy'
    else:
        rf = RandomForestRegressor()
        scoring_metric = 'r2'  # Use R-squared for regression
    
    print(f'Using {rf} to fit, optimising {scoring_metric}')

    rf_search = GridSearchCV(
        estimator=rf, 
        param_grid=random_grid, 
        cv=3,
        scoring=scoring_metric
    )
    
    rf_search.fit(X, y)
    params = rf_search.best_params_
    score = rf_search.best_score_

    # Train the final model with the best params
    if is_classification:
        model = RandomForestClassifier(**params)
    else:
        model = RandomForestRegressor(**params)
    
    model.fit(X, y)
    
    return model, params, score

def cross_fitting_residuals(X, y):
    """
    Performs cross-fitting and computes residuals.
    
    Parameters:
        X (pd.DataFrame or np.array): Feature matrix.
        y (pd.Series or np.array): Target variable.
        
    Returns:
        np.array: Residuals (true values - predicted values).
    """
    n_split = int(X.shape[0] / 2)

    # Ensure correct row selection
    X1, y1 = X.iloc[:n_split], y.iloc[:n_split]
    X2, y2 = X.iloc[n_split:], y.iloc[n_split:]

    model_1, params1, score1 = train_model(X1, y1)
    model_2, params2, score2 = train_model(X2, y2)

    print(f'training score on data1 is {score1} with optimal params :{params1}')
    print(f'training score on data2 is {score2} with optimal params :{params2}')

    is_classification = np.issubdtype(y.dtype, np.number) and y.nunique() == 2

    if is_classification:
        print(f'Using RF classifier to predict')
        predictions = np.concatenate([
            model_1.predict_proba(X2)[:, 1], 
            model_2.predict_proba(X1)[:, 1]
        ])
    else:
        print(f'Using RF regressor to predict, storing residuals')
        predictions = np.concatenate([
            model_1.predict(X2), 
            model_2.predict(X1)
        ])
    
    return y - predictions


In [23]:
print(data_bootstrap['e401'].nunique() == 2,
    len(data_bootstrap['e401'].unique()) == 2,
    np.issubdtype(data_bootstrap['e401'].dtype, np.number) and data_bootstrap['e401'].nunique() == 2)

True True True


In [27]:
X = data_bootstrap[confounders]
y = data_bootstrap[treatment] # decision variable - 'p401' and 'e401 binary, so should use Classifier
residuals_d = cross_fitting_residuals(X, y) # use C into D as independent variable in regression

X = data_bootstrap[confounders]
y = data_bootstrap[outcome] # outcome - 'net_tfa' continuous, so should use Regressor
residuals_y = cross_fitting_residuals(X, y) # use C into D as dependent variable

Unique values in y: [1 0]
Using RandomForestClassifier() to fit, optimising accuracy
Unique values in y: [0 1]
Using RandomForestClassifier() to fit, optimising accuracy
training score on data1 is 0.7038727896537899 with optimal params :{'max_depth': 6, 'n_estimators': 150}
training score on data2 is 0.7094061097008835 with optimal params :{'max_depth': 6, 'n_estimators': 200}
Using RF classifier to predict
Unique values in y: [ 10300.   2078.   1283. ... 484198.  -3290. 110800.]
Using RandomForestRegressor() to fit, optimising r2
Unique values in y: [ 4449.  -600.  -900. ... 16100.  -735. 79699.]
Using RandomForestRegressor() to fit, optimising r2
training score on data1 is 0.4455521399022282 with optimal params :{'max_depth': 6, 'n_estimators': 200}
training score on data2 is 0.39312876597806284 with optimal params :{'max_depth': 6, 'n_estimators': 200}
Using RF regressor to predict, storing residuals


In [28]:
residuals_d_ols = sm.add_constant(residuals_d)
model = sm.OLS(residuals_y, residuals_d_ols)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                net_tfa   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     996.7
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          3.31e-215
Time:                        11:29:53   Log-Likelihood:            -3.7451e+05
No. Observations:               29745   AIC:                         7.490e+05
Df Residuals:                   29743   BIC:                         7.490e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         61.8489    412.170      0.150      0.8